# 🚀 DeBERTa Resume NER Training - Single Cell
## Complete training pipeline in one cell

**Labels from your data:**
- Work Experience: COMPANY, CLIENT, ROLE, LOCATION, DATE_START, DATE_END
- Education: DEGREE, FIELD, INSTITUTION, EDU_YEAR_START, EDU_YEAR_END, GRADE

**Dataset:** 19,292 samples (15,433 train, 1,930 test)

In [ ]:
# ============================================================================
# COMPLETE TRAINING PIPELINE - RUN THIS CELL
# ============================================================================

# 1. Verify GPU
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ NO GPU - STOP HERE")
    raise SystemExit("GPU not enabled. Go to Runtime > Change runtime type > T4 GPU")

# 2. Clean up previous runs
!rm -rf /content/*
print("\n🧹 Cleaned workspace")

# 3. Upload training data ZIP
from google.colab import files
print("\n📤 Upload your Lakshya-Colab-Training.zip file...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# 4. Extract
!unzip -q "{zip_name}" -d /content/
print(f"✅ Extracted {zip_name}")

# 5. Navigate to training directory
%cd /content/Lakshya-Colab-Training/ai-service

# 6. Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q transformers==4.44.0 datasets==2.19.0 accelerate==0.33.0 evaluate==0.4.1 seqeval scikit-learn torch==2.4.0
print("✅ Dependencies installed")

# 7. Start training
print("\n🚀 Starting DeBERTa training...")
print("⏱️  Expected time: ~30-40 minutes on T4 GPU")
print("="*60)
!python training/train.py

# 8. Mount Google Drive to save model
print("\n💾 Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

# 9. Copy trained model to Google Drive
!mkdir -p "/content/drive/MyDrive/Resume-NER-Models"
!cp -r models/resume-ner-deberta "/content/drive/MyDrive/Resume-NER-Models/"
print("\n✅ Model saved to Google Drive: /MyDrive/Resume-NER-Models/resume-ner-deberta")

# 10. Verify saved model
print("\n📂 Model files:")
!ls -lh "/content/drive/MyDrive/Resume-NER-Models/resume-ner-deberta"

# 11. Create download ZIP (optional)
print("\n📦 Creating downloadable ZIP...")
!cd models && zip -r -q resume-ner-deberta.zip resume-ner-deberta
!mv models/resume-ner-deberta.zip /content/

print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print("\n✅ Model saved to:")
print("   1. Google Drive: /MyDrive/Resume-NER-Models/resume-ner-deberta")
print("   2. Download ZIP: Click below to download")
print("\n📥 Downloading model ZIP...")
files.download('/content/resume-ner-deberta.zip')
print("\n✅ Extract and place in: ai-service/models/resume-ner-deberta")

## 🧪 Test the Model (Optional)
Run this cell to test your trained model

In [ ]:
from transformers import pipeline

# Load the trained model
ner_pipeline = pipeline(
    "ner",
    model="models/resume-ner-deberta",
    aggregation_strategy="simple"
)

# Test with sample text
test_text = """Worked at Infosys as Senior Data Engineer in Hyderabad from Jan 2021 to Mar 2023. 
Client was Google. Education: B.Tech in Computer Science from JNTU Hyderabad, 2015-2019, Grade 8.2"""

print("📝 Test Input:")
print(test_text)
print("\n🔍 Predictions:")
print("="*60)

predictions = ner_pipeline(test_text)
for pred in predictions:
    print(f"{pred['entity_group']:20s} | {pred['word']:30s} | Score: {pred['score']:.3f}")

print("\n✅ Model test complete!")